#### Défi quotidien : Analyse stratégique des performances des supermarchés

En tant que **Data Scientist** senior, vous êtes chargé d'analyser un dataset de ventes de supermarchés (Superstore) dans le but de fournir aux décideurs des informations exploitables. Vos objectifs sont multiples :

1. **Comprendre** la structure du jeu de données, le contenu des variables et la qualité des données.

2. **Nettoyer** et prétraiter les données pour les rendre adaptées à l'analyse.

3. **Analyser** les performances des ventes via des visualisations interactives et statiques.

4. **Identifier** les produits, catégories et régions les plus rentables.

5. **Fournir** un rapport exécutif synthétisant vos constats et recommandations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, IntSlider
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings('ignore')

# Configuration esthétique des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

#### Nettoyez et prétraitez vos données :

In [ ]:
# Chargement du dataset US Superstore
df = pd.read_csv('dataset/Sample_Superstore.csv', encoding='latin1')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print('\n--- Info ---')
df.info()
print('\n--- Describe ---')
print(df.describe())
print('\n--- Missing Values ---')
print(df.isnull().sum())

#### Corriger les types de données

In [ ]:
# Suppression des doublons
print(f'Duplicate rows: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'After removal - Shape: {df.shape}')

# Gestion des valeurs manquantes
df['Postal Code'] = df['Postal Code'].fillna(0)
print(f'Postal Code NaN filled with 0')
print(f'Missing values after fill:\n{df.isnull().sum()})'

#### Ingénierie des fonctionnalités

In [ ]:
# Conversion des dates
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')
print(f'Order Date dtype: {df["Order Date"].dtype}')
print(f'Ship Date dtype: {df["Ship Date"].dtype}')

# Création de variables dérivées
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')
print(f'New columns created: Profit Margin, Order Year, Order Month, Order Month-Year')

**2. Analyse exploratoire approfondie (Matplotlib avec ipywidgets)**

In [ ]:
# Graphique interactif : Tendance des ventes mensuelles par catégorie
def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12, 6))
    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values,
                 marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title('Tendance des ventes mensuelles (Toutes catégories)', fontsize=14, fontweight='bold')
    else:
        category_data = df[df['Category'] == category]
        monthly = category_data.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(monthly.index.to_timestamp(), monthly.values,
                 marker='o', linewidth=2, markersize=4, color='seagreen')
        plt.title(f'Tendance des ventes mensuelles — {category}', fontsize=14, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Ventes ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

categories = ['All'] + sorted(df['Category'].unique().tolist())
interact(plot_monthly_sales, category=Dropdown(options=categories, value='All', description='Catégorie:'))

**Performances des ventes géographiques**

In [ ]:
# Graphique interactif : Top N États par ventes
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    top_states = state_sales.tail(top_n)
    bars = plt.barh(range(len(top_states)), top_states.values, color='royalblue')
    plt.yticks(range(len(top_states)), top_states.index)
    plt.xlabel('Total Ventes ($)', fontsize=12)
    plt.ylabel('État', fontsize=12)
    plt.title(f'Top {top_n} États par Ventes', fontsize=14, fontweight='bold')
    for i, (state, value) in enumerate(top_states.items()):
        plt.text(value + max(top_states.values()) * 0.01, i, f'${value:,.0f}',
                 va='center', fontsize=10)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_top_states, top_n=IntSlider(min=5, max=20, value=10, description='Top N:'))

**3. Communiquer des informations (Seaborn)**

In [ ]:
# Top 10 produits les plus rentables
top_products = df.groupby('Product Name')['Profit'].sum().nlargest(10)
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(x=top_products.values, y=top_products.index, ax=ax, palette='viridis')
ax.set_title('Top 10 Produits les plus rentables', fontsize=14, fontweight='bold')
ax.set_xlabel('Bénéfice total ($)', fontsize=12)
ax.set_ylabel('Produit', fontsize=12)
for i, v in enumerate(top_products.values):
    ax.text(v, i, f' ${v:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Résumé exécutif : Les 10 premiers produits génèrent des bénéfices significatifs. '
      'Envisagez d'augmenter les stocks pour ces articles.')

**Diagramme de dispersion : Remise vs Bénéfice**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=60, ax=ax)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red',
            line_kws={'linewidth': 2, 'linestyle': '--'}, ax=ax)
ax.set_title('Analyse : Impact des remises sur les bénéfices par catégorie', fontsize=14, fontweight='bold')
ax.set_xlabel('Taux de remise', fontsize=12)
ax.set_ylabel('Bénéfice ($)', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.legend(title='Catégorie', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('Analyse : Des remises élevées ( > 20%) sont corrélées à des pertes dans plusieurs catégories. '
      'Revoir la stratégie de remises pour préserver la rentabilité.')

**4. Revue de la méthodologie et des outils**

In [ ]:
print('Comparaison des bibliothèques : Pandas pour la manipulation, Matplotlib/Seaborn pour les visualisations statiques, Plotly pour les graphiques interactifs.')
start = time.time()
_ = df.groupby('Category')['Sales'].sum()
end = time.time()
print(f'Temps groupby Pandas : {end - start:.4f} secondes')

**5. Livrable final — Rapport exécutif**

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_margin = df['Profit Margin'].mean()
print(f'Ventes totales : ${total_sales:,.2f}')
print(f'Bénéfice total : ${total_profit:,.2f}')
print(f'Marge bénéficiaire moyenne : {avg_margin:.2f}%')
print('Recommandation : Concentrer les efforts sur les produits les plus performants et optimiser la politique de remises.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ventes par année
df.groupby('Order Year')['Sales'].sum().plot(ax=axes[0, 0], kind='bar', color='steelblue')
axes[0, 0].set_title('Ventes par Année', fontweight='bold')
axes[0, 0].set_ylabel('Ventes ($)')
axes[0, 0].tick_params(axis='x', rotation=0)

# Bénéfice par région
df.groupby('Region')['Profit'].sum().plot(ax=axes[0, 1], kind='bar', color='seagreen')
axes[0, 1].set_title('Bénéfice par Région', fontweight='bold')
axes[0, 1].set_ylabel('Bénéfice ($)')
axes[0, 1].tick_params(axis='x', rotation=0)

# Distribution de la marge bénéficiaire
df['Profit Margin'].hist(ax=axes[1, 0], bins=30, color='coral', edgecolor='black')
axes[1, 0].set_title('Distribution de la Marge Bénéficiaire', fontweight='bold')
axes[1, 0].set_xlabel('Marge (%)')
axes[1, 0].set_ylabel('Fréquence')

# Remise moyenne par catégorie
df.groupby('Category')['Discount'].mean().plot(ax=axes[1, 1], kind='bar', color='gold')
axes[1, 1].set_title('Remise Moyenne par Catégorie', fontweight='bold')
axes[1, 1].set_ylabel('Remise (%)')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

*Annoter les valeurs aberrantes dans le tableau Remise vs. Bénéfice*

In [ ]:
top3 = df.nlargest(3, 'Profit')[['Discount', 'Profit', 'Product Name']]
bottom3 = df.nsmallest(3, 'Profit')[['Discount', 'Profit', 'Product Name']]
print('Top 3 bénéfices (valeurs atypiques positives) :')
print(top3)
print('\nBottom 3 bénéfices (valeurs atypiques négatives) :')
print(bottom3)

*Recréez un graphique interactif avec Plotly Express*

In [ ]:
fig = px.scatter(df, x='Discount', y='Profit', hover_data=['Product Name', 'Category'],
                title='Analyse interactive : Remise vs Bénéfice', 
                labels={'Discount': 'Remise', 'Profit': 'Bénéfice ($)'})
fig.update_layout(xaxis_title='Taux de remise', yaxis_title='Bénéfice ($)')
fig.show()

#### Réponse

In [ ]:
# ==========================================================================
# RÉPONSE COMPLÈTE — Analyse stratégique Superstore
# ==========================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# =====================================================================
# 1. CHARGEMENT ET PRÉTRAITEMENT DES DONNÉES
# =====================================================================
df = pd.read_csv('dataset/Sample_Superstore.csv', encoding='latin1')

print('--- Aperçu du dataset ---')
print(f'Shape : {df.shape}')
print('Colonnes :', df.columns.tolist())

# Suppression des doublons
df = df.drop_duplicates()

# Gestion des valeurs manquantes
df['Postal Code'] = df['Postal Code'].fillna(0)

# Conversion des dates
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')

# Ingénierie des fonctionnalités
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

# =====================================================================
# 2. ANALYSE EXPLORATOIRE — VISUALISATIONS INTERACTIVES
# =====================================================================

# --- Tendance des ventes mensuelles (interactif) ---
def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12, 6))
    if category == 'All':
        total = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total.index.to_timestamp(), total.values, marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title('Tendance des ventes mensuelles (Toutes catégories)', fontsize=14, fontweight='bold')
    else:
        sub = df[df['Category'] == category]
        monthly = sub.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(monthly.index.to_timestamp(), monthly.values, marker='o', linewidth=2, markersize=4, color='seagreen')
        plt.title(f'Tendance des ventes mensuelles — {category}', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Ventes ($)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

cats = ['All'] + sorted(df['Category'].unique().tolist())
interact(plot_monthly_sales, category=Dropdown(options=cats, value='All', description='Catégorie:'))

# --- Top États par ventes (interactif) ---
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    top = state_sales.tail(top_n)
    plt.barh(range(len(top)), top.values, color='royalblue')
    plt.yticks(range(len(top)), top.index)
    plt.xlabel('Ventes totales ($)')
    plt.ylabel('État')
    plt.title(f'Top {top_n} États par Ventes', fontsize=14, fontweight='bold')
    for i, (s, v) in enumerate(top.items()):
        plt.text(v + max(top.values()) * 0.01, i, f'${v:,.0f}', va='center')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_top_states, top_n=IntSlider(min=5, max=20, value=10, description='Top N:'))

# =====================================================================
# 3. ANALYSE DES PRODUITS ET DES REMISES (Seaborn)
# =====================================================================

# --- Top 10 produits rentables ---
top_products = df.groupby('Product Name')['Profit'].sum().nlargest(10)
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(x=top_products.values, y=top_products.index, ax=ax, palette='viridis')
ax.set_title('Top 10 Produits les plus rentables', fontsize=14, fontweight='bold')
ax.set_xlabel('Bénéfice total ($)')
ax.set_ylabel('Produit')
for i, v in enumerate(top_products.values):
    ax.text(v, i, f' ${v:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Résumé exécutif : Les 10 premiers produits génèrent des bénéfices significatifs. '
      'Envisagez d'augmenter les stocks pour ces articles.')

# --- Nuage de points Remise vs Bénéfice ---
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=60, ax=ax)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red',
            line_kws={'linewidth': 2, 'linestyle': '--'}, ax=ax)
ax.set_title('Analyse : Impact des remises sur les bénéfices par catégorie', fontsize=14, fontweight='bold')
ax.set_xlabel('Taux de remise')
ax.set_ylabel('Bénéfice ($)')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.legend(title='Catégorie', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

high_discount = df[df['Discount'] > 0.2]
print('Analyse des remises élevées :')
print(f'• Transactions avec > 20% de remise : {len(high_discount):,}')
print(f'• Bénéfice moyen pour fortes remises : ${high_discount["Profit"].mean():.2f}')
print(f'• % de ventes avec fortes remises générant des pertes : {(high_discount["Profit"] < 0).mean() * 100:.1f}%')

# =====================================================================
# 4. VALEURS ABERRANTES ET GRAPHIQUE INTERACTIF PLOTLY
# =====================================================================

top3 = df.nlargest(3, 'Profit')[['Discount', 'Profit', 'Product Name']]
bottom3 = df.nsmallest(3, 'Profit')[['Discount', 'Profit', 'Product Name']]
print('\nTop 3 bénéfices (valeurs atypiques positives) :')
print(top3.to_string(index=False))
print('\nBottom 3 bénéfices (valeurs atypiques négatives) :')
print(bottom3.to_string(index=False))

fig = px.scatter(df, x='Discount', y='Profit', hover_data=['Product Name', 'Category'],
                title='Analyse interactive : Remise vs Bénéfice (Plotly)',
                labels={'Discount': 'Remise', 'Profit': 'Bénéfice ($)'})
fig.update_layout(xaxis_title='Taux de remise', yaxis_title='Bénéfice ($)')
fig.show()

# =====================================================================
# 5. REVUE MÉTHODOLOGIE ET OUTILS
# =====================================================================

start = time.time()
_ = df.groupby('Category')['Sales'].sum()
end = time.time()
print(f'\nTemps d'exécution Pandas groupby : {end - start:.4f} secondes')
print('\nComparaison des bibliothèques :')
print('• Matplotlib : Contrôle granulaire, widgets ipywidgets, annotations personnalisées.')
print('• Seaborn : Visualisations statistiques prêtes, palettes automatiques, regplot intégré.')
print('• Plotly : Interactivité native (zoom, survol), partage en ligne, tooltips intelligents.')

# =====================================================================
# 6. TABLEAU DE BORD FINAL (DASHBOARD)
# =====================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Order Year')['Sales'].sum().plot(ax=axes[0, 0], kind='bar', color='steelblue')
axes[0, 0].set_title('Ventes par Année', fontweight='bold')
axes[0, 0].set_ylabel('Ventes ($)')
axes[0, 0].tick_params(axis='x', rotation=0)

df.groupby('Region')['Profit'].sum().plot(ax=axes[0, 1], kind='bar', color='seagreen')
axes[0, 1].set_title('Bénéfice par Région', fontweight='bold')
axes[0, 1].set_ylabel('Bénéfice ($)')
axes[0, 1].tick_params(axis='x', rotation=0)

df['Profit Margin'].hist(ax=axes[1, 0], bins=30, color='coral', edgecolor='black')
axes[1, 0].set_title('Distribution de la Marge Bénéficiaire', fontweight='bold')
axes[1, 0].set_xlabel('Marge (%)')
axes[1, 0].set_ylabel('Fréquence')

df.groupby('Category')['Discount'].mean().plot(ax=axes[1, 1], kind='bar', color='gold')
axes[1, 1].set_title('Remise Moyenne par Catégorie', fontweight='bold')
axes[1, 1].set_ylabel('Remise (%)')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# =====================================================================
# RAPPORT EXÉCUTIF
# =====================================================================
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin_pct = (total_profit / total_sales) * 100

print('\n=== RAPPORT EXÉCUTIF — PERFORMANCE SUPERSTORE ===')
print(f'• Ventes totales : ${total_sales:,.0f}')
print(f'• Bénéfice total : ${total_profit:,.0f}')
print(f'• Marge bénéficiaire globale : {profit_margin_pct:.1f}%')

top_state = state_sales.index[-1]
top_state_val = state_sales.iloc[-1]
print(f'• État le plus performant : {top_state} (${top_state_val:,.0f})')
print(f'• Concentration géographique : Top 5 États = {(state_sales.tail(5).sum()/total_sales)*100:.1f}% des ventes')

top_cat = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).index[0]
print(f'• Catégorie leader : {top_cat}')
print(f'• Produit le plus rentable : {top_products.index[0]}')

loss_rate = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100
print(f'• Risque remises élevées : {loss_rate:.1f}% des ventes avec > 20% de remise génèrent des pertes')
print('• Seuil de remise recommandé : 20% pour maintenir la rentabilité')

print('\n--- Recommandations ---')
print('1. Optimiser la politique de remises pour éviter les pertes.')
print('2. Prioriser les stocks dans les États à forte contribution.')
print('3. Mettre en avant les catégories et produits les plus profitables dans les campagnes marketing.')
print('4. Améliorer la saisonnalité des ventes via des promotions ciblées.')
